# LSTM for GeoAI Network Signal Prediction

**Dataset:** `raw_dataset.csv`  
**Target:** next RSRP value (`Level`)  
**Input:** previous 10 observations  
**GeoAI output:** predicted RSRP shown at latitude and longitude.

Run all cells from top to bottom.

## 1. Install libraries

In [ ]:
%pip install -q tensorflow pandas numpy scikit-learn matplotlib folium

## 2. Import libraries

`folium` is the **geospatial library** used to create the interactive map.

In [ ]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import folium  # GEOSPATIAL

SEED=42
LOOKBACK=10
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)

## 3. Load and inspect the dataset

In [ ]:
PATH=Path('raw_dataset.csv')
if not PATH.exists():
    raise FileNotFoundError('Place raw_dataset.csv in the same folder as this notebook.')

df=pd.read_csv(PATH)
print('Dataset shape:', df.shape)
display(df.head())
print(df.columns.tolist())

## 4. Clean and organise the spatio-temporal data

The data are sorted by `SessionID` and `Timestamp` so that LSTM receives the correct time order.

In [ ]:
required=['Timestamp','SessionID','Level','Latitude','Longitude']
missing=[c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df['Timestamp']=pd.to_datetime(df['Timestamp'], format='%Y.%m.%d_%H.%M.%S', errors='coerce')
candidates=['Level','SNR','CQI','DL_bitrate','UL_bitrate','Speed','Latitude','Longitude']
features=[c for c in candidates if c in df.columns]

df=df.dropna(subset=required).copy()
df=df.sort_values(['SessionID','Timestamp']).reset_index(drop=True)
df[features]=df.groupby('SessionID')[features].transform(lambda x:x.ffill().bfill())
df[features]=df[features].fillna(df[features].median(numeric_only=True))

print('Rows after cleaning:', len(df))
print('Input features:', features)
display(df[['Timestamp','SessionID']+features].head())

## 5. Define the prediction task

- **Input:** RSRP, SNR, CQI, DL bitrate, UL bitrate, speed, latitude and longitude from the previous 10 observations.
- **Target:** RSRP (`Level`) at the next observation.

In [ ]:
TARGET='Level'
print('Features:', features)
print('Target:', TARGET)

## 6. Split by drive-test session

In [ ]:
sessions=(df.groupby('SessionID')['Timestamp'].min().sort_values().index.tolist())
if len(sessions)<3:
    raise ValueError('At least 3 sessions are required for train/validation/test splitting.')
n=len(sessions)
a=max(1,int(n*.70)); b=max(a+1,int(n*.85))
train_sessions=sessions[:a]
val_sessions=sessions[a:b]
test_sessions=sessions[b:]
print('Train:',len(train_sessions),'Validation:',len(val_sessions),'Test:',len(test_sessions))

## 7. Scale inputs using training data only

In [ ]:
train_mask=df['SessionID'].isin(train_sessions)
feature_scaler=StandardScaler().fit(df.loc[train_mask,features])
target_scaler=StandardScaler().fit(df.loc[train_mask,[TARGET]])

ds=df.copy()
# Keep raw coordinates for the output map before scaling the model features.
ds['Map_Latitude']=ds['Latitude']
ds['Map_Longitude']=ds['Longitude']
ds[features]=feature_scaler.transform(df[features])
ds['Target_scaled']=target_scaler.transform(df[[TARGET]]).ravel()

## 8. Create LSTM sequences

Each sample uses the previous 10 observations to predict the next RSRP value.

In [ ]:
def make_sequences(data, session_ids, lookback=10):
    X,y,meta=[],[],[]
    for sid,g in data[data['SessionID'].isin(session_ids)].groupby('SessionID'):
        g=g.sort_values('Timestamp').reset_index(drop=True)
        xv=g[features].to_numpy(np.float32)
        yv=g['Target_scaled'].to_numpy(np.float32)
        for i in range(lookback,len(g)):
            X.append(xv[i-lookback:i]); y.append(yv[i])
            meta.append({'SessionID':sid,'Timestamp':g.loc[i,'Timestamp'],
                         'Latitude':g.loc[i,'Map_Latitude'],'Longitude':g.loc[i,'Map_Longitude'],
                         'Actual_RSRP':g.loc[i,TARGET]})
    return np.asarray(X,np.float32),np.asarray(y,np.float32),pd.DataFrame(meta)

X_train,y_train,m_train=make_sequences(ds,train_sessions,LOOKBACK)
X_val,y_val,m_val=make_sequences(ds,val_sessions,LOOKBACK)
X_test,y_test,m_test=make_sequences(ds,test_sessions,LOOKBACK)
print('X_train:',X_train.shape,'X_val:',X_val.shape,'X_test:',X_test.shape)

## 9. Inspect one LSTM training sample

In [ ]:
display(pd.DataFrame(X_train[0],columns=features))
next_rsrp=target_scaler.inverse_transform([[y_train[0]]])[0,0]
print('Target RSRP at next observation:',round(next_rsrp,2),'dBm')

## 10. Build the LSTM model

In [ ]:
model=Sequential([
    Input(shape=(LOOKBACK,len(features))),
    LSTM(64,return_sequences=True),
    Dropout(.2),
    LSTM(32),
    Dropout(.2),
    Dense(16,activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam',loss='mse',metrics=['mae'])
model.summary()

## 11. Train the model

In [ ]:
early=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=30,batch_size=64,callbacks=[early],verbose=1)

## 12. Plot training history

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(history.history['loss'],label='Training loss')
plt.plot(history.history['val_loss'],label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('LSTM Training History')
plt.legend(); plt.grid(True); plt.show()

## 13. Predict the next RSRP values

In [ ]:
pred_scaled=model.predict(X_test,verbose=0)
pred=target_scaler.inverse_transform(pred_scaled).ravel()
actual=target_scaler.inverse_transform(y_test.reshape(-1,1)).ravel()
results=m_test.copy()
results['Predicted_RSRP']=pred
results['Absolute_Error_dB']=np.abs(results['Actual_RSRP']-results['Predicted_RSRP'])
display(results.head(10))

## 14. Evaluate the model

In [ ]:
mae=mean_absolute_error(actual,pred)
rmse=np.sqrt(mean_squared_error(actual,pred))
r2=r2_score(actual,pred)
print(f'MAE: {mae:.3f} dB')
print(f'RMSE: {rmse:.3f} dB')
print(f'R²: {r2:.3f}')

## 15. Plot actual versus predicted RSRP

In [ ]:
n=min(150,len(results))
plt.figure(figsize=(12,5))
plt.plot(results['Actual_RSRP'].iloc[:n].to_numpy(),label='Actual RSRP')
plt.plot(results['Predicted_RSRP'].iloc[:n].to_numpy(),label='Predicted RSRP')
plt.xlabel('Test observation'); plt.ylabel('RSRP (dBm)')
plt.title('Actual vs Predicted RSRP'); plt.legend(); plt.grid(True); plt.show()

## 16. Create the GeoAI prediction map

LSTM predicts the numeric RSRP value. `folium` places that prediction at the corresponding latitude and longitude.

In [ ]:
def colour(v):
    return 'green' if v>=-90 else ('orange' if v>=-105 else 'red')

center=[results['Latitude'].mean(),results['Longitude'].mean()]
m=folium.Map(location=center,zoom_start=14,tiles='OpenStreetMap')
for _,r in results.iloc[:min(500,len(results))].iterrows():
    folium.CircleMarker(
        [r['Latitude'],r['Longitude']],radius=4,color=colour(r['Predicted_RSRP']),
        fill=True,fill_opacity=.75,
        popup=(f"Time: {r['Timestamp']}<br>Actual: {r['Actual_RSRP']:.2f} dBm<br>"
               f"Predicted: {r['Predicted_RSRP']:.2f} dBm<br>Error: {r['Absolute_Error_dB']:.2f} dB")
    ).add_to(m)
m.save('lstm_geoai_rsrp_map.html')
m

## 17. Save outputs

In [ ]:
results.to_csv('lstm_rsrp_predictions.csv',index=False)
model.save('lstm_rsrp_model.keras')
print('Saved predictions, model, and GeoAI map.')

## Teaching flow

```text
raw_dataset.csv
      ↓
clean and sort by time
      ↓
scale variables
      ↓
create 10-step sequences
      ↓
train LSTM
      ↓
predict next RSRP
      ↓
evaluate and display on GeoAI map
```